# 統一Factory実践：モデルから候補点提案まで

このNotebookで確認すること：公開名の探索、既存クラスとの同一性、明示的な学習、posterior、獲得関数、OptimizerとSearch Strategyの違い、Mixed・制約・多目的・分類ALへの応用。

[Theory](../../docs/theory/13_model_selection.md) / [Guide](../../docs/guide/13_model_selection.md) / [利用方法](../../docs/capability_workflow.md)

Data → Model → Fit → Posterior → Acquisition → Optimization → Candidate → Evaluation

合成データを使用し、回帰BOでは最大化を行います。分類ALでは目的値ではなくラベルの不確実性を評価します。

In [ ]:
import torch
import matplotlib.pyplot as plt
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.utils.multi_objective.box_decompositions.non_dominated import (
    FastNondominatedPartitioning,
)
from robotorchan.models import SingleTaskGP, create_model, get_model_class, list_models
from robotorchan.acquisition import (
    create_acquisition, get_acquisition_class, list_acquisitions,
)
from robotorchan.optim import (
    create_optimizer, create_search_strategy, list_optimizers, list_search_strategies,
)
from robotorchan.optim.constraints.contracts import CandidateConstraints
from robotorchan.optim.domains.variable_space import MixedVariableSpace

torch.manual_seed(17)
dtype = torch.double

## 1. 公開名とコンストラクタ

一覧は実装の発見に使います。名前の存在だけでは組合せの実行可能性を保証しません。Factoryは既存のクラスを返し、独自の代理モデルを作りません。

In [ ]:
for names in (list_models(), list_acquisitions(), list_optimizers(), list_search_strategies()):
    print(len(names), list(names)[:8])
assert get_model_class("SingleTaskGP") is SingleTaskGP
assert get_acquisition_class("qLogExpectedImprovement") is qLogExpectedImprovement
print(list_optimizers())
print(list_search_strategies())

## 2. 単目的回帰：データ・学習・予測

入力は n × d、観測は n × 1。真の目的関数は $f(x)=-(x-0.65)^2$ です。Factoryはfitを呼びません。直接生成との型の一致を確認してから学習します。

In [ ]:
X = torch.linspace(0.1, 0.9, 9, dtype=dtype).unsqueeze(-1)
Y = -((X - 0.65) ** 2)
bounds = torch.tensor([[0.0], [1.0]], dtype=dtype)
model = create_model("SingleTaskGP", X, Y)
direct = SingleTaskGP(X, Y)
assert type(model) is type(direct)
model.fit(optimizer_kwargs={"options": {"maxiter": 30}})
grid = torch.linspace(0, 1, 101, dtype=dtype).unsqueeze(-1)
with torch.no_grad():
    posterior = model.posterior(grid)
    mean = posterior.mean.squeeze(-1)
    std = posterior.variance.clamp_min(0).sqrt().squeeze(-1)
print(X.shape, Y.shape, posterior.mean.shape, posterior.variance.shape)

## 3. 獲得関数とq-batch

best_fは既観測の最大値。qLogExpectedImprovementは改善量の期待値を安定な対数表現で扱うMC獲得関数です。評価入力は batch × q × d。q=1の曲線とq=2の同時提案は別の計算です。

In [ ]:
acq = create_acquisition(
    "qLogExpectedImprovement", model=model, best_f=Y.max(),
    sampler=SobolQMCNormalSampler(torch.Size([32]), seed=17),
)
with torch.no_grad():
    scores = acq(grid.unsqueeze(-2))
print(scores.shape)
optimize = create_optimizer("sobol", raw_samples=64, seed=17)
candidate, value = optimize(acq, bounds, q=2)
assert candidate.shape == (2, 1)
assert torch.isfinite(candidate).all()
print(candidate, value, -((candidate - 0.65) ** 2))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
axes[0].plot(grid.squeeze(), mean, label="Posterior mean")
axes[0].fill_between(grid.squeeze(), mean - 1.96 * std, mean + 1.96 * std, alpha=0.2)
axes[0].scatter(X.squeeze(), Y.squeeze(), label="Observations")
axes[0].scatter(candidate.squeeze(), -((candidate.squeeze() - 0.65) ** 2), label="Candidates")
axes[0].legend()
axes[1].plot(grid.squeeze(), scores, label="q=1 log EI")
axes[1].legend()
axes[1].set_xlabel("x")
plt.tight_layout()
plt.show()

## 4. Search Strategy Factory

Optimizer Factoryは関数を返すのに対し、Search Strategy Factoryは探索空間を持つインスタンスを返します。ここではSobolを使い、同じ獲得関数から候補を生成します。

In [ ]:
strategy = create_search_strategy("SobolSearchStrategy", bounds, num_samples=64, seed=17)
result = strategy.optimize(acq, q=2)
assert result.candidates.shape == (2, 1)
print(result.candidates, result.acquisition_value)

## 5. Mixed入力と候補制約

第2変数は整数0/1。モデルのcat_dimsはカテゴリカーネルの指定であり、Optimizer側のinteger_dimsは候補の許容値を指定します。線形不等式は x[0] >= 0.2 です。

In [ ]:
Xm = torch.cat([X, (torch.arange(len(X)) % 2).to(dtype).unsqueeze(-1)], dim=-1)
Ym = Y + 0.1 * Xm[:, 1:]
bm = torch.tensor([[0., 0.], [1., 1.]], dtype=dtype)
mixed = create_model("MixedSingleTaskGP", Xm, Ym, cat_dims=[1])
mixed.fit(optimizer_kwargs={"options": {"maxiter": 10}})
am = create_acquisition("qLogExpectedImprovement", model=mixed, best_f=Ym.max())
space = MixedVariableSpace(bounds=bm, integer_dims=(1,))
constraints = CandidateConstraints(
    inequality_constraints=((torch.tensor([0]), torch.tensor([1.], dtype=dtype), 0.2),),
)
cm, vm = optimize(am, bm, q=2, variable_space=space, constraints=constraints)
assert cm.shape == (2, 2)
assert torch.all(cm[:, 0] >= 0.2 - 1e-8)
assert torch.equal(cm[:, 1], cm[:, 1].round())
print(cm, vm)

## 6. 多目的回帰

2目的を同時に最大化します。参照点は観測の各目的の最小値より低く設定します。qLogEHVIはPareto集合のhypervolume改善を評価します。

In [ ]:
Y2 = torch.cat([Y, -((X - 0.25) ** 2)], dim=-1)
m2 = create_model("SingleTaskGP", X, Y2)
m2.fit(optimizer_kwargs={"options": {"maxiter": 10}})
ref = Y2.min(dim=0).values - 0.1
partition = FastNondominatedPartitioning(ref_point=ref, Y=Y2)
a2 = create_acquisition(
    "qLogExpectedHypervolumeImprovement", model=m2, ref_point=ref.tolist(),
    partitioning=partition, sampler=SobolQMCNormalSampler(torch.Size([16]), seed=17),
)
c2, v2 = optimize(a2, bounds, q=2)
y2_new = torch.cat([-((c2 - 0.65) ** 2), -((c2 - 0.25) ** 2)], dim=-1)
assert c2.shape == (2, 1)
print(c2, v2, y2_new)
plt.scatter(Y2[:, 0], Y2[:, 1], label="Observations")
plt.scatter(y2_new[:, 0], y2_new[:, 1], label="Candidates")
plt.xlabel("Objective 1")
plt.ylabel("Objective 2")
plt.legend()
plt.show()

## 7. 二値・多クラス・順序分類のAL

クラスラベルを数値回帰へ変換しません。各分類モデルをfitし、predict_probaで予測クラス確率を確認します。qPredictiveEntropyはラベルの不確実性に基づく獲得関数で、目的値の最大化とは異なります。短い学習はAPI確認用であり、精度比較ではありません。

In [ ]:
for name, k in [
    ("BinarySingleTaskGPClassifier", 2),
    ("MulticlassSingleTaskGPClassifier", 3),
    ("OrdinalSingleTaskGPClassifier", 3),
]:
    labels = (X.squeeze(-1) * k).long()
    kwargs = {"num_classes": k} if k > 2 else {}
    classifier = create_model(name, X, labels, **kwargs)
    classifier.fit(epochs=2)
    with torch.no_grad():
        probabilities = classifier.predict_proba(grid)
    entropy = create_acquisition("qPredictiveEntropy", model=classifier)
    proposed = strategy.optimize(entropy, q=2).candidates
    assert proposed.shape == (2, 1)
    print(name, probabilities.shape, proposed.flatten().tolist())
    plt.figure(figsize=(7, 3))
    plt.plot(grid.squeeze(), probabilities)
    for x in proposed.flatten():
        plt.axvline(x.item(), linestyle="--")
    plt.title(name)
    plt.xlabel("x")
    plt.ylabel("Class probability")
    plt.show()

## 8. 注意点と次のステップ

Factoryは名前解決と引数の受け渡しを担当します。モデル固有の引数、posteriorの意味、制約、q対応は各APIの契約に従います。ここでは代表的な構成を実行し、全モデルの組合せを保証するものではありません。次のNotebookでは能力ベースの選択とStudyによる反復・履歴復元を扱います。

参考：Rasmussen and Williams (2006), Gaussian Processes for Machine Learning; Frazier (2018), A Tutorial on Bayesian Optimization.